# 01 · Multimodal Foundations

> **Source notes:** `MultimodalFoundations.md`

How do images, audio, and video become tensors a neural network can process?

This notebook makes it concrete:
- **Image tensors** — load a JPEG, inspect shape, normalise, visualise channels
- **Audio tensors** — build a mel spectrogram from a synthetic waveform
- **Video tensors** — sample frames from a video and inspect the 4-D tensor shape
- **The modality gap** — show why you cannot simply concatenate image and text tensors

**All local, no API key or GPU needed.** Runtime: < 30 seconds.

## 0 · Setup

```bash
# No Ollama needed for this chapter.
# Packages installed below via pip.
```

In [ ]:
# TODO: Implement this cell


## 1 · Image Tensors

A colour image in PyTorch is a tensor of shape **(C, H, W)** — channels first.

| Dimension | Meaning | Typical size |
|-----------|---------|-------------|
| C         | Colour channels (R, G, B) | 3 |
| H         | Height in pixels | 224 (ViT standard) |
| W         | Width in pixels | 224 |

**Normalisation** converts uint8 $[0, 255]$ to float32 and shifts the distribution so each channel has zero mean and unit variance (ImageNet statistics).

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 2 · Audio Tensors — Mel Spectrograms

Audio models almost never work on raw waveforms. They convert the signal to a **mel spectrogram** — a 2-D frequency-vs-time representation that:
- Compresses the time axis (from 16,000 samples/sec to ~100 frames/sec)
- Uses a perceptual frequency scale (mel) that matches human hearing
- Applies log compression to handle dynamic range

The result looks like an image, which is why ViT-based models (Whisper, AST) can process audio directly.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 3 · Video Tensors

Video extends images by adding a time dimension. A video tensor has shape **(T, C, H, W)**:

- `T` = number of sampled frames (not all frames — uniform or random sampling)
- `C, H, W` = channels, height, width of each frame

The key challenge: even a 1-second clip at 30fps = 30 frames × 3 × 224 × 224 ≈ 4.5 million values. Models sample a small subset.

In [ ]:
# TODO: Implement this cell


## 4 · The Modality Gap

This cell shows *why* you cannot simply concatenate image and text tensors. The distributions are fundamentally different — they don't live in the same space.

In [ ]:
# TODO: Implement this cell


## 5 · Summary — PixelSmith v0

```
┌─────────────────────────────────────────────────────────────────────────────┐
│ PixelSmith v0 — Input stage                                                  │
│                                                                               │
│  JPEG / PNG  ──load──▶  (H, W, C) uint8                                     │
│                ──CHW──▶  (C, H, W) float32 [0,1]                            │
│                ──norm──▶  (C, H, W) ImageNet-normalised  OR  [-1,1]         │
│                ──batch─▶  (N, C, H, W)                ← ready for Ch.2 ViT │
│                                                                               │
│  Audio .wav ──STFT──▶  (1, F, T') log-mel-spec  ← treated as 2-D image     │
│  Video .mp4 ──sample──▶  (T, C, H, W) per-frame float32                    │
└─────────────────────────────────────────────────────────────────────────────┘
```

**Key takeaways:**
1. All modalities become tensors of floats — but with different shapes and value ranges.
2. Normalisation convention matters: use `-1 to 1` for diffusion, ImageNet stats for ViT/CLIP/ResNet.
3. The modality gap means a shared embedding space is required — which CLIP (Ch.3) provides.
4. Audio models work on mel spectrograms because they look like 2-D images.

**Next:** [VisionTransformers.md](../VisionTransformers/VisionTransformers.md) — how the ViT splits your `(3, H, W)` tensor into patches and applies self-attention.